# Analise de Renda Fixa — Tesouro Direto

Analise de titulos publicos, curva de juros reais e comparativo com benchmarks.

Fontes:
- **TesouroDiretoFetcher**: taxas atuais, historico NTN-B, curva de juros reais
- **BCBFetcher**: Selic, CDI, IPCA (contexto para renda fixa)

**Nota**: Nao requer API keys. Dados publicos do Tesouro Nacional e BCB.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Taxas Atuais — Tesouro Direto

In [ ]:
from carteira_auto.data.fetchers import TesouroDiretoFetcher

tesouro = TesouroDiretoFetcher()

# Taxas atuais de todos os titulos
taxas = tesouro.get_current_rates()
print(f"Titulos disponiveis: {len(taxas)}")
print()
print(taxas.to_string(index=False))

## 2. Curva NTN-B — Juros Reais

A curva NTN-B mostra as taxas reais (acima do IPCA) para diferentes vencimentos.
Util para avaliar o premio de risco e a expectativa de inflacao.

In [ ]:
# Curva NTN-B (juros reais por vencimento)
curva = tesouro.get_ntnb_curve()
print("Curva NTN-B (juros reais):")
print(curva)

if not curva.empty:
    fig, ax = plt.subplots()
    ax.plot(curva.index, curva.values, "o-", markersize=8, linewidth=2, color="#d62728")
    ax.set_title("Curva de Juros Reais (NTN-B)")
    ax.set_xlabel("Vencimento")
    ax.set_ylabel("Taxa real (% a.a.)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 3. Historico NTN-B Principal

In [ ]:
# Historico de precos da NTN-B Principal
ntnb = tesouro.get_ntnb_history(com_cupom=False)
print(f"NTN-B Principal historico: {len(ntnb)} registros")
if not ntnb.empty:
    print(ntnb.tail(5))

## 4. Contexto — Selic, CDI e IPCA

Para avaliar renda fixa, comparamos com os benchmarks de referencia.

In [ ]:
from carteira_auto.data.fetchers import BCBFetcher

bcb = BCBFetcher()

selic = bcb.get_selic(period_days=30)
ipca = bcb.get_ipca(period_days=365)

selic_atual = selic["valor"].iloc[-1]
ipca_12m = ((1 + ipca.tail(12)["valor"] / 100).prod() - 1) * 100

print(f"Selic meta: {selic_atual:.2f}% a.a.")
print(f"IPCA acumulado 12m: {ipca_12m:.2f}%")
print(f"Juro real (Selic - IPCA): {selic_atual - ipca_12m:.2f}% a.a.")

In [ ]:
# Comparativo: NTN-B vs Selic real
print("\n=== Comparativo de Rentabilidade ===")
print(f"Selic nominal: {selic_atual:.2f}% a.a.")
print(f"Selic real (- IPCA): {selic_atual - ipca_12m:.2f}% a.a.")
if not curva.empty:
    ntnb_longa = curva.iloc[-1]
    print(f"NTN-B longa (juro real): {ntnb_longa:.2f}% a.a.")
    print(f"\nNTN-B longa vs Selic real: {ntnb_longa - (selic_atual - ipca_12m):+.2f} pp")

## Resumo

| Titulo | Tipo | Referencia | Melhor para |
|--------|------|------------|-------------|
| Tesouro Selic | Pos-fixado | Selic | Liquidez, reserva de emergencia |
| Tesouro IPCA+ | Inflacao + real | IPCA + taxa | Protecao inflacionaria, longo prazo |
| Tesouro Prefixado | Taxa fixa | CDI | Cenario de queda de juros |